Download classification dataset:

https://archive.ics.uci.edu/dataset/53/iris


Data definition, from iris.names:

7. Attribute Information:
   1. sepal length in cm (sepal_length)
   2. sepal width in cm (sepal_width)
   3. petal length in cm (petal_length)
   4. petal width in cm (petal_width)
   5. class: ==> class
      -- Iris Setosa
      -- Iris Versicolour
      -- Iris Virginica


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split


In [3]:
column_names=["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]
data = pd.read_csv('data/iris.data', names=column_names)

# make species a categorical column
data['species'] = data['species'].astype('category')

X = data.drop(["species"], axis=1) 
y = pd.get_dummies(data["species"], dtype=int)


Let's make sure the shapes are what we expected:

In [4]:
assert(X.shape == (150,4))
assert(y.shape == (150,3))

Let's look at the input data

In [5]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
dtypes: float64(4)
memory usage: 4.8 KB


In [6]:
# y now has evenly-divided categories -- one of each type
y.value_counts()

Iris-setosa  Iris-versicolor  Iris-virginica
0            0                1                 50
             1                0                 50
1            0                0                 50
Name: count, dtype: int64

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 

Size of inputs
20% are reserved for testing


In [8]:
print("total number of examples   ", len(X))
print("number of training examples", len(X_train))
print("number of test examples    ", len(X_test))

total number of examples    150
number of training examples 120
number of test examples     30


Sample training data


In [9]:
print("** X_train info")
X_train.info()

print("\n** X_train values[0]")
print(X_train.values[0])

print("]\n** y info")
y_train.info()


** X_train info
<class 'pandas.core.frame.DataFrame'>
Index: 120 entries, 22 to 102
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  120 non-null    float64
 1   sepal_width   120 non-null    float64
 2   petal_length  120 non-null    float64
 3   petal_width   120 non-null    float64
dtypes: float64(4)
memory usage: 4.7 KB

** X_train values[0]
[4.6 3.6 1.  0.2]
]
** y info
<class 'pandas.core.frame.DataFrame'>
Index: 120 entries, 22 to 102
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Iris-setosa      120 non-null    int64
 1   Iris-versicolor  120 non-null    int64
 2   Iris-virginica   120 non-null    int64
dtypes: int64(3)
memory usage: 3.8 KB


In [10]:
print(X_train.iloc[0:5])

    sepal_length  sepal_width  petal_length  petal_width
22           4.6          3.6           1.0          0.2
15           5.7          4.4           1.5          0.4
65           6.7          3.1           4.4          1.4
11           4.8          3.4           1.6          0.2
42           4.4          3.2           1.3          0.2


In [11]:
from micrograd.engine import Value
from micrograd.nn import Neuron, Layer, MLP

In [12]:
model = MLP(4, [8, 8, 3]) # 3-layer neural network
print(model)
print("number of parameters", len(model.parameters()))

Multi-Layer Perceptron Structure:
 Layer 1/3 - Shape of the layer is: 4 X 8 (nin X nout)
	[Neuron 0: ReLUNeuron(4) -> w0 =-0.2764, w1 = 0.4989, w2 = 0.1106, w3 =-0.3246, b = 0.0000 ... Neuron 7: ReLUNeuron(4) -> w0 =-0.0633, w1 = 0.2629, w2 =-0.2524, w3 = 0.6384, b = 0.0000]
 Layer 2/3 - Shape of the layer is: 8 X 8 (nin X nout)
	[Neuron 0: ReLUNeuron(8) -> w0 = 0.6402, w1 = 0.3944, w2 = 0.4498, w3 = 0.7164, w4 =-0.8148, w5 = 0.3007, w6 = 0.5246, w7 =-0.7289, b = 0.0000 ... Neuron 7: ReLUNeuron(8) -> w0 = 0.0513, w1 =-0.3764, w2 = 0.9726, w3 = 0.2407, w4 = 0.7646, w5 = 0.8334, w6 =-0.6531, w7 =-0.8685, b = 0.0000]
 Layer 3/3 - Shape of the layer is: 8 X 3 (nin X nout)
	[Neuron 0: LinearNeuron(8) -> w0 =-0.2673, w1 = 0.0936, w2 = 0.6899, w3 =-0.7407, w4 =-0.4488, w5 = 0.5986, w6 =-0.0426, w7 =-0.1106, b = 0.0000 ... Neuron 2: LinearNeuron(8) -> w0 = 0.0489, w1 =-0.1859, w2 = 0.2547, w3 =-0.5654, w4 =-0.0009, w5 =-0.6156, w6 =-0.8818, w7 =-0.1813, b = 0.0000]

number of parameters 139


In [13]:
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [14]:
import math 

def softmax(z):  
    """ Softmax converts a vector of values to a probability distribution.
    Args:
      z (ndarray (N,))  : input data, N features
    Returns:
      a (ndarray (N,))  : softmax of z
    """    
    ### START CODE HERE ### 
    l = np.array(z)
    e_pow_z = math.e**l
    e_pow_z_sum = np.sum(e_pow_z)
    a = e_pow_z / e_pow_z_sum
    return a


In [15]:
lst = [-2, .2, 1.6, 3.2]
softmax(lst)

array([0.00438798, 0.03960155, 0.16059218, 0.79541829])

In [32]:
def cross_entropy(y_pred, y_true):
 
    # computing softmax values for predicted values
    y_pred = softmax(y_pred)
    loss = 0
     
    # Doing cross entropy Loss
    for i in range(len(y_pred)):
 
        # Here, the loss is computed using the
        # above mathematical formulation.
        loss = loss + (-1 * y_true[i]*np.log(y_pred[i]))
 
    return loss
 

In [33]:
X_train_np = X_train.values
y_train_np = y_train.values

assert X_train_np.shape == (120, 4)
assert y_train_np.shape == (120, 3)


In [35]:
# loss function
def loss(batch_size=None):

    # inline DataLoader :)
    if batch_size is None:
        Xb, yb = X_train_np, y_train_np
    else:
        ri = np.random.permutation(X.shape[0])[:batch_size]
        Xb, yb = X_train_np[ri], y_train_np[ri]
    inputs = [list(map(Value, xrow)) for xrow in Xb]
    
    # forward the model to get scores
    scores = list(map(model, inputs))
    
    def get_data(v):
        return [value.data for value in v]
    
    raw_scores = list(map(get_data, scores))

    soft_scores = [softmax(scores_output) for scores_output in raw_scores]
    # print(f'**  raw_scores: {raw_scores[0]}')
    # print(f'** soft_scores: {soft_scores[0]}')
    data_loss = cross_entropy(np.array(soft_scores), yb)
    # L2 regularization
    alpha = 1e-4
    reg_loss = alpha * sum((p*p for p in model.parameters()))
    total_loss = data_loss + reg_loss
    
    # also get accuracy
    accuracy = [yi ==  np.argmax(scores) for yi, scores in zip(yb, raw_scores)]
    return total_loss, sum(accuracy) / len(accuracy)

total_loss, acc = loss()
print("initial total_loss: ", total_loss, " activation: ", acc)

initial total_loss:  [Value(data=228.85132950845335, grad=0)
 Value(data=226.5657097782827, grad=0)
 Value(data=244.3276300811205, grad=0)]  activation:  [0.05       0.625      0.60833333]


In [36]:
# optimization
for k in range(100):
    
    # forward
    total_loss, acc = loss()
    
    # backward
    model.zero_grad()
    total_loss.backward()
    
    # update (sgd)
    learning_rate = 5.0 - 0.9*k/100    
    for p in model.parameters():
        p.data -= learning_rate * p.grad
    
    if k % 5 == 0:
        print(f"step {k} loss {total_loss.data}, accuracy {acc*100}%")


AttributeError: 'numpy.ndarray' object has no attribute 'backward'